In [ ]:
pip install linearmodels

In [ ]:
import pandas as pd
import numpy as np

from linearmodels.panel import PanelOLS, RandomEffects
from linearmodels.panel import compare

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


import plotly.graph_objects as go
import plotly.express as px

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_path = '/content/drive/My Drive/Colab Notebooks/phd_data/final_data.csv'

df = pd.read_csv(file_path, sep=';', decimal=',')
df.head()

,Country Name,Country Code,Region,Religion,Year,GMI,Fertility,Age Dependency,Armed Forces Pers,Corruption,...,Life Expectancy,Military Expenditure GDP,Military Expenditure Gov Exp,Political Stability,PARL,Regulatory Quality,Rule of Law,Unemployment,Voice and Accountability,Women Business
0,Ukraine,UKR,Eastern Europe,Christian,1990,NaN,58.033,50.48832,NaN,NaN,...,70.09756,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55.000
1,Israel,ISR,Middle East,Jewish,1990,386.79136,20.344,67.62692,NaN,NaN,...,76.60732,12.36269,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.125
2,Armenia,ARM,Eastern Europe,Christian,1990,NaN,73.201,54.92935,NaN,NaN,...,68.82100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,58.125
3,Qatar,QAT,Middle East,Muslim,1990,295.09948,52.569,39.57314,NaN,NaN,...,73.98200,10.74388,25.25222,NaN,NaN,NaN,NaN,NaN,NaN,29.375
4,Bahrain,BHR,Middle East,Muslim,1990,244.36899,23.845,56.15405,NaN,NaN,...,72.66800,5.28479,14.28571,NaN,NaN,NaN,NaN,NaN,NaN,26.250


In [ ]:
# Количество стран с GMI за каждый год
gmi_data_per_year = df.groupby('Year')['GMI'].apply(lambda x: x.notnull().sum())

# Количество стран
total_countries_per_year = df.groupby('Year')['Country Name'].nunique()

gmi_data_percentage_per_year = (gmi_data_per_year / total_countries_per_year) * 100

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=gmi_data_percentage_per_year.index,
    y=gmi_data_percentage_per_year,
    mode='lines+markers',
    name='Процент стран с данными по GMI'
))

fig.update_layout(
    title='Процент стран с данными по GMI за каждый год',
    xaxis_title='Год',
    yaxis_title='Процент стран с данными по GMI',
    template='plotly_white'
)

fig.show()

In [ ]:
df = df[df['Year'] >= 1993]
df.head()

,Country Name,Country Code,Region,Religion,Year,GMI,Fertility,Age Dependency,Armed Forces Pers,Corruption,...,Life Expectancy,Military Expenditure GDP,Military Expenditure Gov Exp,Political Stability,PARL,Regulatory Quality,Rule of Law,Unemployment,Voice and Accountability,Women Business
486,Ukraine,UKR,Eastern Europe,Christian,1993,135.56920,61.192,51.51111,1.56594,NaN,...,68.13073,0.35235,NaN,NaN,NaN,NaN,NaN,2.000,NaN,66.875
487,Israel,ISR,Middle East,Jewish,1993,377.11522,19.157,65.35243,8.03536,NaN,...,77.15366,11.52324,NaN,NaN,NaN,NaN,NaN,12.741,NaN,65.625
488,Armenia,ARM,Eastern Europe,Christian,1993,185.77991,85.981,58.75430,1.35950,NaN,...,68.84100,2.29664,NaN,NaN,NaN,NaN,NaN,5.300,NaN,58.125
489,Qatar,QAT,Middle East,Muslim,1993,317.65976,40.221,38.51663,2.72872,NaN,...,74.24300,NaN,NaN,NaN,NaN,NaN,NaN,0.870,NaN,29.375
490,Bahrain,BHR,Middle East,Muslim,1993,295.99251,20.920,55.55339,2.81206,NaN,...,73.37500,5.36986,16.77316,NaN,NaN,NaN,NaN,0.943,NaN,26.250


In [ ]:
df.nunique()

,0
Country Name,162
Country Code,162
Region,13
Religion,7
Year,30
GMI,4487
Fertility,4778
Age Dependency,4855
Armed Forces Pers,4337
Corruption,3848


In [ ]:
country_years_with_gmi = df.groupby('Country Name')['GMI'].apply(lambda x: x.notnull().sum())

total_years_per_country = df['Year'].nunique()
country_gmi_percentage = (country_years_with_gmi / total_years_per_country) * 100

# Разделение на группы по проценту данных
bins = np.arange(0, 110, 10)  # Группы по 10%: 0-10%, 10-20%, ..., 90-100%
country_gmi_bins = pd.cut(country_gmi_percentage, bins=bins)

# Подсчет количества стран в каждой группе
hist_gmi = country_gmi_bins.value_counts().sort_index()

# Построение гистограммы с использованием plotly
fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f"{int(interval.left)}%-{int(interval.right)}%" for interval in hist_gmi.index],
    y=hist_gmi,
    marker_color='lightskyblue',
    name='Процент стран'
))

fig.update_layout(
    title='Процент стран с разным уровнем наличия данных по GMI',
    xaxis_title='Процент наличия данных по GMI',
    yaxis_title='Количество стран',
    template='plotly_white'
)

fig.show()

In [ ]:
# GMI > 60%
countries_with_sufficient_gmi = country_gmi_percentage[country_gmi_percentage > 60].index

df = df[df['Country Name'].isin(countries_with_sufficient_gmi)]
df.head()

,Country Name,Country Code,Region,Religion,Year,GMI,Fertility,Age Dependency,Armed Forces Pers,Corruption,...,Life Expectancy,Military Expenditure GDP,Military Expenditure Gov Exp,Political Stability,PARL,Regulatory Quality,Rule of Law,Unemployment,Voice and Accountability,Women Business
486,Ukraine,UKR,Eastern Europe,Christian,1993,135.56920,61.192,51.51111,1.56594,NaN,...,68.13073,0.35235,NaN,NaN,NaN,NaN,NaN,2.000,NaN,66.875
487,Israel,ISR,Middle East,Jewish,1993,377.11522,19.157,65.35243,8.03536,NaN,...,77.15366,11.52324,NaN,NaN,NaN,NaN,NaN,12.741,NaN,65.625
488,Armenia,ARM,Eastern Europe,Christian,1993,185.77991,85.981,58.75430,1.35950,NaN,...,68.84100,2.29664,NaN,NaN,NaN,NaN,NaN,5.300,NaN,58.125
490,Bahrain,BHR,Middle East,Muslim,1993,295.99251,20.920,55.55339,2.81206,NaN,...,73.37500,5.36986,16.77316,NaN,NaN,NaN,NaN,0.943,NaN,26.250
491,Saudi Arabia,SAU,Middle East,Muslim,1993,281.41425,56.751,74.60796,3.05151,NaN,...,70.30300,12.46537,32.83411,NaN,NaN,NaN,NaN,7.075,NaN,23.750


In [ ]:
# PARL > 60%
country_years_with_parl = df.groupby('Country Name')['PARL'].apply(lambda x: x.notnull().sum())

country_parl_percentage = (country_years_with_parl / total_years_per_country) * 100
countries_with_sufficient_parl = country_parl_percentage[country_parl_percentage > 60].index

# фильтрация df
df = df[df['Country Name'].isin(countries_with_sufficient_parl)]
df.head()

,Country Name,Country Code,Region,Religion,Year,GMI,Fertility,Age Dependency,Armed Forces Pers,Corruption,...,Life Expectancy,Military Expenditure GDP,Military Expenditure Gov Exp,Political Stability,PARL,Regulatory Quality,Rule of Law,Unemployment,Voice and Accountability,Women Business
486,Ukraine,UKR,Eastern Europe,Christian,1993,135.56920,61.192,51.51111,1.56594,NaN,...,68.13073,0.35235,NaN,NaN,NaN,NaN,NaN,2.000,NaN,66.875
487,Israel,ISR,Middle East,Jewish,1993,377.11522,19.157,65.35243,8.03536,NaN,...,77.15366,11.52324,NaN,NaN,NaN,NaN,NaN,12.741,NaN,65.625
488,Armenia,ARM,Eastern Europe,Christian,1993,185.77991,85.981,58.75430,1.35950,NaN,...,68.84100,2.29664,NaN,NaN,NaN,NaN,NaN,5.300,NaN,58.125
490,Bahrain,BHR,Middle East,Muslim,1993,295.99251,20.920,55.55339,2.81206,NaN,...,73.37500,5.36986,16.77316,NaN,NaN,NaN,NaN,0.943,NaN,26.250
491,Saudi Arabia,SAU,Middle East,Muslim,1993,281.41425,56.751,74.60796,3.05151,NaN,...,70.30300,12.46537,32.83411,NaN,NaN,NaN,NaN,7.075,NaN,23.750


In [ ]:
# GNE > 60%
country_years_with_gne = df.groupby('Country Name')['GNE'].apply(lambda x: x.notnull().sum())

country_gne_percentage = (country_years_with_gne / total_years_per_country) * 100
countries_with_sufficient_gne = country_gne_percentage[country_gne_percentage > 60].index

# фильтрация df
df = df[df['Country Name'].isin(countries_with_sufficient_gne)]
df.head()

,Country Name,Country Code,Region,Religion,Year,GMI,Fertility,Age Dependency,Armed Forces Pers,Corruption,...,Life Expectancy,Military Expenditure GDP,Military Expenditure Gov Exp,Political Stability,PARL,Regulatory Quality,Rule of Law,Unemployment,Voice and Accountability,Women Business
486,Ukraine,UKR,Eastern Europe,Christian,1993,135.56920,61.192,51.51111,1.56594,NaN,...,68.13073,0.35235,NaN,NaN,NaN,NaN,NaN,2.000,NaN,66.875
487,Israel,ISR,Middle East,Jewish,1993,377.11522,19.157,65.35243,8.03536,NaN,...,77.15366,11.52324,NaN,NaN,NaN,NaN,NaN,12.741,NaN,65.625
490,Bahrain,BHR,Middle East,Muslim,1993,295.99251,20.920,55.55339,2.81206,NaN,...,73.37500,5.36986,16.77316,NaN,NaN,NaN,NaN,0.943,NaN,26.250
491,Saudi Arabia,SAU,Middle East,Muslim,1993,281.41425,56.751,74.60796,3.05151,NaN,...,70.30300,12.46537,32.83411,NaN,NaN,NaN,NaN,7.075,NaN,23.750
492,Greece,GRC,Western Europe,Christian,1993,234.37930,15.886,48.17184,4.95763,NaN,...,77.39024,3.60012,8.11759,NaN,NaN,NaN,NaN,10.177,NaN,70.625


In [ ]:
# Military Expenditure Gov Exp > 60%
country_years_with_mil_exp_gov_exp = df.groupby('Country Name')['Military Expenditure Gov Exp'].apply(lambda x: x.notnull().sum())

country_mil_exp_gov_exp_percentage = (country_years_with_mil_exp_gov_exp / total_years_per_country) * 100
countries_with_sufficient_mil_exp_gov_exp = country_mil_exp_gov_exp_percentage[country_mil_exp_gov_exp_percentage > 60].index

# фильтрация df
df = df[df['Country Name'].isin(countries_with_sufficient_mil_exp_gov_exp)]
df.head()

,Country Name,Country Code,Region,Religion,Year,GMI,Fertility,Age Dependency,Armed Forces Pers,Corruption,...,Life Expectancy,Military Expenditure GDP,Military Expenditure Gov Exp,Political Stability,PARL,Regulatory Quality,Rule of Law,Unemployment,Voice and Accountability,Women Business
486,Ukraine,UKR,Eastern Europe,Christian,1993,135.56920,61.192,51.51111,1.56594,NaN,...,68.13073,0.35235,NaN,NaN,NaN,NaN,NaN,2.000,NaN,66.875
487,Israel,ISR,Middle East,Jewish,1993,377.11522,19.157,65.35243,8.03536,NaN,...,77.15366,11.52324,NaN,NaN,NaN,NaN,NaN,12.741,NaN,65.625
490,Bahrain,BHR,Middle East,Muslim,1993,295.99251,20.920,55.55339,2.81206,NaN,...,73.37500,5.36986,16.77316,NaN,NaN,NaN,NaN,0.943,NaN,26.250
491,Saudi Arabia,SAU,Middle East,Muslim,1993,281.41425,56.751,74.60796,3.05151,NaN,...,70.30300,12.46537,32.83411,NaN,NaN,NaN,NaN,7.075,NaN,23.750
492,Greece,GRC,Western Europe,Christian,1993,234.37930,15.886,48.17184,4.95763,NaN,...,77.39024,3.60012,8.11759,NaN,NaN,NaN,NaN,10.177,NaN,70.625


**Сверху кончается обработка данных**

In [ ]:
# Создание нового DataFrame с усредненными значениями GMI по регионам и годам
grouped_df_gmi = df.groupby(['Region', 'Year'])['GMI'].mean().reset_index()

In [ ]:
fig = go.Figure()

# Группировка данных по региону и году и усреднение GMI
for region, group in grouped_df_gmi.groupby('Region'):
    # средний GMI по региону
    avg_gmi_per_year = group.groupby('Year')['GMI'].mean()

    # Добавление линии на график для каждого региона
    fig.add_trace(
        go.Scatter(
            x=avg_gmi_per_year.index,
            y=avg_gmi_per_year.values,
            mode='lines',
            name=f"{region}"
        )
    )

# Настройки оси X
fig.update_xaxes(
    tickmode='linear',
    dtick=1,
    title="Year",
    rangeslider=dict(visible=True),
    type="linear"
)

# Настройки оси Y
fig.update_yaxes(
    title="Average GMI"
)

# Настройки макета графика
fig.update_layout(
    title="Средний GMI по годам для каждого региона",
    width=1200,
    height=800,
    template='plotly_white'
)

# Показ графика
fig.show()

In [ ]:
# Создание нового DataFrame с усредненными значениями PARL по регионам и годам
grouped_df_parl = df.groupby(['Region', 'Year'])['PARL'].mean().reset_index()

In [ ]:
# Создание пустого графика
fig = go.Figure()

# Группировка данных по региону и году и усреднение PARL
for region, group in grouped_df_parl.groupby('Region'):
    # Расчет среднего PARL по годам внутри региона
    avg_parl_per_year = group.groupby('Year')['PARL'].mean()

    # Добавление линии на график для каждого региона
    fig.add_trace(
        go.Scatter(
            x=avg_parl_per_year.index,
            y=avg_parl_per_year.values,
            mode='lines',
            name=f"{region}"
        )
    )

# Настройки оси X
fig.update_xaxes(
    tickmode='linear',
    dtick=1,
    title="Year",
    rangeslider=dict(visible=True),
    type="linear"
)

# Настройки оси Y
fig.update_yaxes(
    title="Average PARL"
)

# Настройки макета графика
fig.update_layout(
    title="Средний PARL по годам для каждого региона",
    width=1200,
    height=800,
    template='plotly_white'
)

# Показ графика
fig.show()

**Далее строим модели**

In [ ]:
df_cleaned = df.dropna()
df_cleaned = df_cleaned.drop(columns = ['Military Expenditure GDP', 'Military Expenditure Gov Exp'])
df_cleaned.head()

NameError: name 'df' is not defined

In [ ]:
# Матрица корреляций

correlation_matrix = df_cleaned.corr()

fig = px.imshow(
    correlation_matrix,
    text_auto=True,  # Показывает значения корреляции
    color_continuous_scale='Viridis',  # Цветовая палитра
    title="Матрица корреляций переменных",
    labels=dict(color="Коэффициент корреляции"),
    width=800,
    height=800
)

fig.update_layout(
    xaxis=dict(tickangle=45),  # Угол подписей оси X
    template="plotly_white"
)

fig.show()

NameError: name 'df_cleaned' is not defined

In [ ]:
region_results = {}
for region in df_cleaned['Region'].unique():
  # Фильтрация данных по региону
  regional_data = df_cleaned[df_cleaned['Region'] == region]

  # Установка индексов
  regional_data = regional_data.set_index(['Country Name', 'Year'])

  # Указание зависимой и независимых переменных
  dependent = regional_data['GMI']
  independent = regional_data.drop(columns=['GMI', 'Country Code', 'Region', 'Religion'])

  try:
    model = PanelOLS(dependent, independent, entity_effects=True)
    result = model.fit()
    print(region)
    print(result.summary)

  except Exception as e:
    print(f"Error in {region}: {e}")

Western Europe
                          PanelOLS Estimation Summary                           
Dep. Variable:                    GMI   R-squared:                        0.5035
Estimator:                   PanelOLS   R-squared (Between):             -7.6294
No. Observations:                 416   R-squared (Within):               0.5035
Date:                Sun, Dec 01 2024   R-squared (Overall):             -7.5507
Time:                        15:44:25   Log-likelihood                   -1513.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      24.089
Entities:                          20   P-value                           0.0000
Avg Obs:                       20.800   Distribution:                  F(16,380)
Min Obs:                       19.000                                           
Max Obs:                       21.000   F-statistic (robust):             24.089
             

In [ ]:
religion_results = {}
for religion in df_cleaned['Religion'].unique():
  # Фильтрация данных по религии
  religional_data = df_cleaned[df_cleaned['Religion'] == religion]

  # Установка индексов
  religional_data = religional_data.set_index(['Country Name', 'Year'])

  # Указание зависимой и независимых переменных
  dependent = religional_data['GMI']
  independent = religional_data.drop(columns=['GMI', 'Country Code', 'Region', 'Religion'])

  try:
    model = PanelOLS(dependent, independent, entity_effects=True)
    result = model.fit()
    print(religion)
    print(result.summary)

  except Exception as e:
    print(f"Error in {religion}: {e}")

Christian
                          PanelOLS Estimation Summary                           
Dep. Variable:                    GMI   R-squared:                        0.4097
Estimator:                   PanelOLS   R-squared (Between):             -0.6168
No. Observations:                1599   R-squared (Within):               0.4097
Date:                Sun, Dec 01 2024   R-squared (Overall):             -0.5784
Time:                        15:44:36   Log-likelihood                   -6060.6
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      65.148
Entities:                          81   P-value                           0.0000
Avg Obs:                       19.741   Distribution:                 F(16,1502)
Min Obs:                       13.000                                           
Max Obs:                       21.000   F-statistic (robust):             65.148
                  

In [ ]:
# Убираем ненужные столбцы и пропуски
data_for_clustering = df_cleaned.drop(columns=['Country Name', 'Country Code', 'Region', 'Religion'])
data_for_clustering = data_for_clustering.dropna()

# Стандартизация данных
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_for_clustering)

NameError: name 'df_cleaned' is not defined

In [ ]:
inertia = []
range_n_clusters = range(1, 11)  # Пробуем от 1 до 10 кластеров
for n_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(data_scaled)
    inertia.append(kmeans.inertia_)

# Построение графика метода локтя
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range_n_clusters),
    y=inertia,
    mode='lines+markers',
    marker=dict(size=8),
    line=dict(width=2),
    name="Inertia"
))

fig.update_layout(
    title="Метод локтя для выбора числа кластеров",
    xaxis_title="Количество кластеров",
    yaxis_title="Сумма квадратов расстояний (Inertia)",
    template="plotly_white",
    width=800,
    height=500
)

fig.show()

In [ ]:
# Кластеризация K-средних
optimal_clusters = 4
kmeans = KMeans(n_clusters=optimal_clusters, random_state=42)
data_for_clustering['Cluster'] = kmeans.fit_predict(data_scaled)

# Анализ кластеров
df_cleaned['Cluster'] = data_for_clustering['Cluster']
df_cleaned.head()

In [ ]:
cluster_results = {}
for cluster in df_cleaned['Cluster'].unique():
  # Фильтрация данных по религии
  clusteral_data = df_cleaned[df_cleaned['Cluster'] == cluster]

  # Установка индексов
  clusteral_data = clusteral_data.set_index(['Country Name', 'Year'])

  # Указание зависимой и независимых переменных
  dependent = clusteral_data['GMI']
  independent = clusteral_data.drop(columns=['GMI', 'Country Code', 'Region', 'Religion'])

  try:
    model = PanelOLS(dependent, independent, entity_effects=True)
    result = model.fit()
    print(f"Its cluster # {cluster}")
    print(result.summary)

  except Exception as e:
    print(f"Error in {cluster}: {e}")

Its cluster # 1
                          PanelOLS Estimation Summary                           
Dep. Variable:                    GMI   R-squared:                        0.6078
Estimator:                   PanelOLS   R-squared (Between):             -1.0704
No. Observations:                 367   R-squared (Within):               0.6078
Date:                Sun, Dec 01 2024   R-squared (Overall):             -0.6575
Time:                        15:45:01   Log-likelihood                   -1325.7
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      30.698
Entities:                          34   P-value                           0.0000
Avg Obs:                       10.794   Distribution:                  F(16,317)
Min Obs:                       1.0000                                           
Max Obs:                       21.000   F-statistic (robust):             30.698
            